# Cognitive Kernel v0.1 — Training Notebook

Run this in Google Colab. Resumes automatically if a checkpoint exists in `OUTPUT_DIR`.


In [ ]:
# Cell 1: mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cell 2: clone repo (or pull if already cloned)
import os, subprocess
REPO_DIR = '/content/cognitive-kernel'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Jeova-Luks/cognitive-kernel.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)


In [ ]:
# Cell 3: install dependencies
!pip install -q -r requirements.txt


In [ ]:
# Cell 4: wandb login (skip with empty key for offline mode)
import wandb
# wandb.login(key='YOUR_KEY')  # uncomment and paste, or run `wandb login` interactively


In [ ]:
# Cell 5: pick config and output dir
from pathlib import Path
from config import load_config
from trainer import Trainer

CONFIG = 'configs/base_100m.yaml'
OUTPUT_DIR = Path('/content/drive/MyDrive/cognitive-kernel/checkpoints/base_100m')

cfg = load_config(Path(CONFIG))
print(cfg)


In [ ]:
# Cell 6: train (resumes automatically if checkpoint exists in OUTPUT_DIR)
trainer = Trainer(cfg, output_dir=OUTPUT_DIR)
print(f'Model parameters: {sum(p.numel() for p in trainer.model.parameters()):,}')
trainer.fit()


## Loss curves

Reads `metrics.jsonl` from OUTPUT_DIR (written by the trainer) and plots
train_loss + val_loss vs step. Run this cell after Cell 6 finishes.


In [ ]:
# Cell 7: plot loss curves from metrics.jsonl
import json
import matplotlib.pyplot as plt

metrics_path = OUTPUT_DIR / 'metrics.jsonl'
if not metrics_path.exists():
    print(f'No metrics.jsonl found at {metrics_path}; run Cell 6 first.')
else:
    records = [json.loads(l) for l in metrics_path.read_text().splitlines() if l.strip()]
    train_pts = [(r['step'], r['train_loss']) for r in records if 'train_loss' in r]
    val_pts   = [(r['step'], r['val_loss'])   for r in records if 'val_loss'   in r]
    if not train_pts:
        print('No train_loss records yet.')
    else:
        fig, ax = plt.subplots(figsize=(11, 5))
        ts, ls = zip(*train_pts)
        ax.plot(ts, ls, label='train_loss', color='tab:blue', alpha=0.7, linewidth=1)
        if val_pts:
            vs, vl = zip(*val_pts)
            ax.plot(vs, vl, label='val_loss', color='tab:red', marker='o', linestyle='--')
        ax.set_xlabel('step')
        ax.set_ylabel('loss')
        ax.set_title(f'Training curve ({len(records)} log points, final step {trainer.step})')
        ax.legend()
        ax.grid(alpha=0.3)
        plt.show()
        print(f'\\n  First train_loss: {ls[0]:.4f}')
        print(f'  Last  train_loss: {ls[-1]:.4f}')
        print(f'  Delta: {ls[-1] - ls[0]:+.4f}  ({"falling \u2713" if ls[-1] < ls[0] else "rising \u2717"})')
